<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/01-Basic_Tutor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building a Basic AI Tutor with the OpenAI Responses API

This notebook demonstrates how to build a simple AI tutor using the OpenAI Responses API. The tutor is specialized to answer artificial intelligence-related questions and politely declines unrelated topics.

## What This Notebook Covers
- Calling the OpenAI Responses API (`client.responses.create`) with system instructions
- Defining a reusable tutor function that handles errors gracefully
- Filtering responses by topic using system-level guidance
- Maintaining multi-turn conversation history
- (Optional) Streaming responses token-by-token for lower perceived latency
- (Optional) Comparing different tutor personas via system prompt variations

## Learning Objectives
By the end of this notebook, you will be able to:
1. Initialize an OpenAI client and call the Responses API
2. Use `instructions` to constrain model behavior to a specific domain
3. Pass conversation history as a list of messages for multi-turn dialogue
4. Implement streaming responses with `client.responses.stream()`
5. Experiment with different system prompts to shape model personality

## Prerequisites
- Basic Python knowledge
- An OpenAI API key (set as a Colab Secret named `OPENAI_API_KEY`, or in a local `.env` file)

# Install Packages and Setup Variables


In [ ]:
!pip install -q openai==2.8.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 13.2 MB/s eta 0:00:00


In [ ]:
import os

# Load the OpenAI API key from Colab Secrets when running on Google Colab,
# or fall back to a local environment variable for local execution.
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
except ImportError:
    os.environ["OPENAI_API_KEY"] = os.environ.get('OPENAI_API_KEY', '')

# Initialize the OpenAI Client

In [ ]:
from openai import OpenAI

# Create the client object that connects to the OpenAI API endpoints.
client = OpenAI()

# Query the API

The `ask_ai_tutor` function below wraps `client.responses.create` to send a question to the model. It passes a system-level `instructions` string that restricts the tutor to AI-related topics only. We then test it with two questions — one on-topic (AI frameworks) and one off-topic (geography) — to verify that the topic filter works correctly.

In [ ]:
# Define two test questions: one AI-related and one on an unrelated topic.
# These will be used to verify the tutor's topic-filtering behavior.
QUESTION_AI = "List a number of famous artificial intelligence frameworks?"
QUESTION_NOT_AI = (
    "What is the name of the highest mountain in the world and its height?"
)

In [ ]:
def ask_ai_tutor(question):
    try:
        # System instructions that restrict the tutor to AI-related questions only
        instructions = (
            "You are an AI tutor specialized in answering artificial intelligence-related questions. "
            "Only answer AI-related questions; otherwise, say that you cannot answer this question."
        )

        # Call the OpenAI Responses API with the question as the user input
        response = client.responses.create(
            model="gpt-5.4-mini",
            reasoning={'effort': 'minimal'},
            instructions=instructions,
            input=f"Please provide an informative and accurate answer to the following question.\nQuestion: {question}\nAnswer:",
        )

        # Return the model's response text
        return response.output_text.strip()

    except Exception as e:
        return f"An error occurred: {e}"

In [ ]:
# Ask the AI-related question — the tutor should answer this.
RES_AI = ask_ai_tutor(QUESTION_AI)
print(RES_AI)

An error occurred: Error code: 400 - {'error': {'message': "Unsupported value: 'minimal' is not supported with the 'gpt-5.4-mini' model. Supported values are: 'none', 'low', 'medium', 'high', and 'xhigh'.", 'type': 'invalid_request_error', 'param': 'reasoning.effort', 'code': 'unsupported_value'}}


In [ ]:
# Ask the off-topic question — the tutor should decline to answer this.
RES_NOT_AI = ask_ai_tutor(QUESTION_NOT_AI)
print(RES_NOT_AI)

An error occurred: Error code: 400 - {'error': {'message': "Unsupported value: 'minimal' is not supported with the 'gpt-5.4-mini' model. Supported values are: 'none', 'low', 'medium', 'high', and 'xhigh'.", 'type': 'invalid_request_error', 'param': 'reasoning.effort', 'code': 'unsupported_value'}}


## Optional: Streaming Responses

By default, the Responses API returns the full answer only after the model has finished generating it. **Streaming** changes this behavior: tokens are delivered as they are produced, so the user sees output appear progressively rather than waiting for the entire response.

This significantly improves **perceived latency** — especially for long answers — and is the pattern used in most production chat interfaces. The `client.responses.stream()` context manager handles the streaming connection and delivers events that you iterate over, extracting text deltas from `response.output_text.delta` events.

In [ ]:
with client.responses.stream(
    model="gpt-5.4-mini",
    reasoning={'effort': 'low'},
    instructions="You are an AI tutor specialized in AI topics.",
    input="What is gradient descent? Explain briefly.",
) as stream:
    print("Streaming response: ", end="", flush=True)
    for event in stream:
        # response.output_text.delta carries each text fragment as it arrives
        if hasattr(event, "type") and event.type == "response.output_text.delta":
            print(event.delta, end="", flush=True)
    print()  # newline at end
    final = stream.get_final_response()
    print(f"\nTotal tokens used: {final.usage.total_tokens}")

Streaming response: Gradient descent is an optimization algorithm used to find the minimum of a function, often the loss function in machine learning.

Briefly:
- Start with an initial guess for the parameters.
- Compute the gradient, which points in the direction of steepest increase.
- Move a small step in the opposite direction of the gradient to reduce the loss.
- Repeat until the loss is small or stops improving.

In formula form:
\[
\theta \leftarrow \theta - \eta \nabla L(\theta)
\]
where:
- \(\theta\) = model parameters
- \(\eta\) = learning rate
- \(\nabla L(\theta)\) = gradient of the loss

It’s the core method used to train many machine learning models.

Total tokens used: 187


# Multi-Turn Conversation with History

The Responses API supports multi-turn conversations by passing the full message history as a list. Each entry in the list has a `role` (`user` or `assistant`) and `content`. This allows the model to refer back to earlier exchanges when answering follow-up questions.

In [ ]:
response = client.responses.create(
    model="gpt-5.4-mini",
    reasoning={'effort': 'low'},
    instructions="You are an AI tutor specialized in answering artificial intelligence-related questions. Only answer AI-related questions; otherwise, say that you cannot answer this question.",
    input=[
        {
            "role": "user",
            "content": "Please provide an informative and accurate answer to the following question.\nQuestion: List a number of famous artificial intelligence frameworks?\nAnswer:",
        },
        {
            "role": "assistant",
            "content": RES_AI,
        },
        {
            "role": "user",
            "content": "Please provide an informative and accurate answer to the following question.\nQuestion: What is the name of the highest mountain in the world and its height?\nAnswer:",
        },
        {
            "role": "assistant",
            "content": RES_NOT_AI,
        },
        {
            "role": "user",
            "content": "Please provide an informative and accurate answer to the following question.\nQuestion: Can you write a summary of the first suggested AI framework in the first question?\nAnswer:",
        },
    ]
)

print(response.output_text.strip())

I can’t reliably summarize the “first suggested AI framework” because the earlier answer wasn’t provided successfully, so no framework was actually listed.

If you want, I can instead give a short summary of a famous AI framework such as:
- TensorFlow
- PyTorch
- Keras
- scikit-learn
- JAX


## Optional: Comparing Different Tutor Personas

The `instructions` field acts as a **system prompt** — it shapes the model's personality, tone, and teaching style without the user ever seeing it. Small changes to the system prompt can produce very different learning experiences.

Below, we ask the same question ("What is a neural network?") using three different personas:
1. **Concise tutor** — gives brief, direct answers
2. **Socratic tutor** — guides the learner with questions rather than giving direct answers
3. **Beginner-friendly tutor** — uses simple language and analogies

This demonstrates how prompt engineering lets you tune model behavior without changing any other code.

In [ ]:
QUESTION_PERSONA = "What is a neural network?"

personas = [
    {
        "label": "Concise Tutor",
        "instructions": (
            "You are a concise AI tutor. Answer questions briefly and directly "
            "in 2-3 sentences. No elaboration unless asked."
        ),
    },
    {
        "label": "Socratic Tutor",
        "instructions": (
            "You are a Socratic AI tutor. Instead of giving direct answers, "
            "guide the learner to the answer by asking thoughtful follow-up questions."
        ),
    },
    {
        "label": "Beginner-Friendly Tutor",
        "instructions": (
            "You are a beginner-friendly AI tutor. Use simple language, everyday analogies, "
            "and avoid technical jargon. Your goal is to make complex ideas easy to grasp."
        ),
    },
]

for persona in personas:
    response = client.responses.create(
        model="gpt-5.4-mini",
        reasoning={'effort': 'low'},
        instructions=persona["instructions"],
        input=f"Question: {QUESTION_PERSONA}\nAnswer:",
    )
    print(f"--- {persona['label']} ---")
    print(response.output_text.strip())
    print()

--- Concise Tutor ---
A neural network is a type of machine learning model inspired by the brain, made of interconnected layers of nodes called neurons. It learns patterns from data by adjusting the connections between these nodes, allowing it to make predictions or classifications.

--- Socratic Tutor ---
What do you think the brain does when it learns from examples?

A neural network is a computer model inspired by that idea: it takes inputs, processes them through layers of connected units, and learns patterns so it can make predictions or decisions.

Can you think of a task where spotting patterns from many examples would be useful?

--- Beginner-Friendly Tutor ---
A **neural network** is a type of computer program designed to learn patterns, a bit like how the human brain learns from experience.

Think of it like a big team of tiny decision-makers working together:
- Each one looks at a small part of the problem.
- They pass information along to each other.
- Over time, the networ